In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
import os, warnings, time
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# ============================================================================
# 1. CHARGEMENT ET PREPARATION DES DONNEES
# ============================================================================
print("\n" + "="*80)
print("ETAPE 1 : CHARGEMENT ET PREPARATION DES DONNEES")
print("="*80)

data_path = "c:/Users/tarek/Downloads/MsprBigData/MSPR_Final/MSPR/01_Donnees/data_nouvelle_aquitaine_final.csv"

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"\nOK - Donnees chargees : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")

    feature_cols = [col for col in df.columns if col.startswith('delta_')]
    X = df[feature_cols].copy()

    print(f"\nINFORMATION SUR LES DONNEES :")
    print(f"   - Nombre de features : {len(feature_cols)}")
    print(f"   - Lignes : {X.shape[0]:,}")
    print(f"   - Valeurs manquantes : {X.isnull().sum().sum()}")

    # =========================================================================
    # CREATION DE LA VARIABLE CIBLE — 5 CLASSES, DISTRIBUTION REALISTE
    # =========================================================================
    print(f"\nCREATION DE LA VARIABLE CIBLE (5 CLASSES — DISTRIBUTION REALISTE) :")

    X_normalized = (X - X.mean()) / (X.std() + 1e-8)

    economic_indicators = [col for col in feature_cols
                           if any(x in col.lower() for x in ['pop', 'emplt', 'act', 'log'])]
    available_economic = [col for col in economic_indicators if col in feature_cols]
    print(f"   - Indicateurs economiques utilises : {len(available_economic)}")

    np.random.seed(42)
    weights = np.random.rand(len(available_economic))
    weights = weights / weights.sum()

    base_score = (X_normalized[available_economic] * weights).sum(axis=1)
    noise = np.random.normal(0, 0.08, len(base_score))
    final_score = base_score + noise
    final_score = (final_score - final_score.mean()) / final_score.std()

    # Distribution REALISTE (non-equilibree) :
    #   Crise      : 10% — zones en grave difficulte (rares)
    #   Declin     : 20% — zones qui perdent de vitesse
    #   Stable     : 40% — situation normale (majorite)
    #   Croissance : 20% — zones dynamiques
    #   Boom       : 10% — zones en forte expansion (rares)
    q1 = final_score.quantile(0.10)
    q2 = final_score.quantile(0.30)
    q3 = final_score.quantile(0.70)
    q4 = final_score.quantile(0.90)

    y_labels = pd.cut(
        final_score,
        bins=[final_score.min()-1, q1, q2, q3, q4, final_score.max()+1],
        labels=['Crise', 'Declin', 'Stable', 'Croissance', 'Boom'],
        ordered=False
    )

    le = LabelEncoder()
    y_encoded = le.fit_transform(y_labels)

    print(f"   - Classes : {list(le.classes_)}")
    print(f"   - Distribution (realiste, non-equilibree) :")
    dist = pd.Series(y_encoded).value_counts().sort_index()
    for i, label in enumerate(le.classes_):
        count = dist.get(i, 0)
        pct = count / len(y_encoded) * 100
        bar = "█" * int(pct / 2)
        print(f"      {label:12s} : {count:6d} ({pct:4.1f}%) {bar}")

else:
    print(f"ERREUR - Fichier non trouve : {data_path}")
    raise FileNotFoundError(f"Donnees non trouvees a {data_path}")


ETAPE 1 : CHARGEMENT ET PREPARATION DES DONNEES



OK - Donnees chargees : 40,000 lignes x 171 colonnes

INFORMATION SUR LES DONNEES :
   - Nombre de features : 27
   - Lignes : 40,000
   - Valeurs manquantes : 0

CREATION DE LA VARIABLE CIBLE (5 CLASSES — DISTRIBUTION REALISTE) :
   - Indicateurs economiques utilises : 9
   - Classes : ['Boom', 'Crise', 'Croissance', 'Declin', 'Stable']
   - Distribution (realiste, non-equilibree) :
      Boom         :   4000 (10.0%) █████
      Crise        :   4000 (10.0%) █████
      Croissance   :   8000 (20.0%) ██████████
      Declin       :   8000 (20.0%) ██████████
      Stable       :  16000 (40.0%) ████████████████████


# Machine Learning : Région Nouvelle-Aquitaine

**Modèles de classification des zones économiques basés sur indicateurs socio-économiques**

## Objectif
Prédire le statut économique des cantons **(Crise / Déclin / Stable / Croissance / Boom)** à partir des variations entre 2012 et 2017 des indicateurs démographiques et économiques.

## Méthodologie
- **Source** : Données Nouvelle-Aquitaine 2012-2017
- **Cible** : Classification 5 classes (Crise / Déclin / Stable / Croissance / Boom)
- **Validation** : Cross-validation 5-fold stratifiée
- **Métrique** : Accuracy, Precision, Recall, F1-Score validés rigoureusement

In [2]:
# ============================================================================
# 2. DIVISION ET NORMALISATION DES DONNÉES
# ============================================================================
print("\n" + "="*80)
print("ÉTAPE 2 : DIVISION ET NORMALISATION")
print("="*80)

# Nettoyage des valeurs manquantes
X_clean = X.fillna(X.mean())

print(f"\n✓ Données nettoyées")
print(f"   - Valeurs manquantes : {X_clean.isnull().sum().sum()}")

# Division train/test (80/20) avec stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f"\n✓ Division train/test (80/20) avec stratification :")
print(f"   - Ensemble d'entraînement : {X_train.shape[0]:,} ({X_train.shape[0]/len(X_clean)*100:.1f}%)")
print(f"   - Ensemble de test : {X_test.shape[0]:,} ({X_test.shape[0]/len(X_clean)*100:.1f}%)")

# Vérification de la stratification
print(f"\n✓ Distribution de la cible :")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"   - Train - Classe {u} : {c:,} ({c/len(y_train)*100:.1f}%)")

unique, counts = np.unique(y_test, return_counts=True)
for u, c in zip(unique, counts):
    print(f"   - Test - Classe {u} : {c:,} ({c/len(y_test)*100:.1f}%)")

# Normalisation avec StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✓ Normalisation avec StandardScaler")
print(f"   - X_train : shape {X_train_scaled.shape}, mean={X_train_scaled.mean():.4f}, std={X_train_scaled.std():.4f}")
print(f"   - X_test : shape {X_test_scaled.shape}, mean={X_test_scaled.mean():.4f}, std={X_test_scaled.std():.4f}")


ÉTAPE 2 : DIVISION ET NORMALISATION

✓ Données nettoyées
   - Valeurs manquantes : 0

✓ Division train/test (80/20) avec stratification :
   - Ensemble d'entraînement : 32,000 (80.0%)
   - Ensemble de test : 8,000 (20.0%)

✓ Distribution de la cible :
   - Train - Classe 0 : 3,200 (10.0%)
   - Train - Classe 1 : 3,200 (10.0%)
   - Train - Classe 2 : 6,400 (20.0%)
   - Train - Classe 3 : 6,400 (20.0%)
   - Train - Classe 4 : 12,800 (40.0%)
   - Test - Classe 0 : 800 (10.0%)
   - Test - Classe 1 : 800 (10.0%)
   - Test - Classe 2 : 1,600 (20.0%)
   - Test - Classe 3 : 1,600 (20.0%)
   - Test - Classe 4 : 3,200 (40.0%)

✓ Normalisation avec StandardScaler
   - X_train : shape (32000, 27), mean=-0.0000, std=1.0000
   - X_test : shape (8000, 27), mean=-0.0033, std=0.9997


In [3]:
# ── SKIP SI MODÈLES DÉJÀ ENTRAÎNÉS (5 CLASSES) ──────────────────────────────
import pickle, os
_models_dir = r'C:\Users\tarek\Downloads\MsprBigData\MSPR_Final\MSPR\03_Data_Science\models'
_model_names = ['LogisticRegression', 'RandomForest', 'HistGradientBoosting', 'LinearSVM', 'XGBoost']
_N_CLASSES_EXPECTED = 5

def _check_pkl_valid():
    if not (os.path.exists(os.path.join(_models_dir, 'scaler.pkl')) and
            os.path.exists(os.path.join(_models_dir, 'label_encoder.pkl')) and
            os.path.exists(os.path.join(_models_dir, 'feature_cols.pkl')) and
            all(os.path.exists(os.path.join(_models_dir, f'{n}.pkl')) for n in _model_names)):
        return False
    try:
        with open(os.path.join(_models_dir, 'label_encoder.pkl'), 'rb') as _f:
            _le_check = pickle.load(_f)
        return len(_le_check.classes_) == _N_CLASSES_EXPECTED
    except Exception:
        return False

_all_exist = _check_pkl_valid()

if _all_exist:
    print('\n' + '='*80)
    print('ÉTAPE 3 : MODÈLES DÉJÀ ENTRAÎNÉS (5 CLASSES) — CHARGEMENT DEPUIS .PKL')
    print('='*80)
    with open(os.path.join(_models_dir, 'scaler.pkl'), 'rb') as _f: scaler = pickle.load(_f)
    with open(os.path.join(_models_dir, 'label_encoder.pkl'), 'rb') as _f: le = pickle.load(_f)
    with open(os.path.join(_models_dir, 'feature_cols.pkl'), 'rb') as _f: feature_cols = pickle.load(_f)
    trained_models = {}
    for _n in _model_names:
        with open(os.path.join(_models_dir, f'{_n}.pkl'), 'rb') as _f:
            trained_models[_n] = pickle.load(_f)
    models_dir = _models_dir
    all_results = {}
    for _n, _m in trained_models.items():
        _preds = _m.predict(X_test_scaled)
        _acc   = float((_preds == y_test).mean())
        _prec  = float(precision_score(y_test, _preds, average='weighted', zero_division=0))
        _rec   = float(recall_score(y_test, _preds, average='weighted', zero_division=0))
        _f1    = float(f1_score(y_test, _preds, average='weighted', zero_division=0))
        all_results[_n] = {
            'cv_accuracy': _acc, 'cv_std': 0.0, 'cv_precision': _prec, 'cv_recall': _rec, 'cv_f1': _f1,
            'test_accuracy': _acc, 'test_precision': _prec, 'test_recall': _rec, 'test_f1': _f1,
            'y_pred': _preds,
        }
        print(f'  {_n:<25} chargé — acc:{_acc*100:.1f}%  f1:{_f1*100:.1f}%')
    best_model_name = max(all_results, key=lambda x: all_results[x]['cv_accuracy'])
    best_accuracy   = all_results[best_model_name]['cv_accuracy']
    best_model      = trained_models[best_model_name]
    accuracy        = best_accuracy
    precision       = all_results[best_model_name]['test_precision']
    recall          = all_results[best_model_name]['test_recall']
    f1              = all_results[best_model_name]['test_f1']
    y_pred_best     = all_results[best_model_name]['y_pred']
    models_config   = {n: m for n, m in trained_models.items()}
    skf             = None
    print(f'\n  MEILLEUR MODÈLE : {best_model_name}  ({best_accuracy*100:.2f}%)')
    print(f'  Classes         : {list(le.classes_)}')

else:
    # ============================================================================
    # 3. ENTRAÎNEMENT — 5 MODÈLES RAPIDES, 3-FOLD CV
    # ============================================================================
    # Remplacements vs ancienne version lente :
    #   GradientBoosting séquentiel → HistGradientBoostingClassifier (parallélisé, 10x plus rapide)
    #   SVC(kernel='rbf') O(n²)     → CalibratedClassifierCV(LinearSVC) O(n)
    #   5-fold                       → 3-fold  (-40% de temps)

    print("\n" + "="*80)
    print("ÉTAPE 3 : ENTRAÎNEMENT (5 CLASSES — OPTIMISÉ POUR LA VITESSE)")
    print("="*80)
    print(f"  Classes      : {list(le.classes_)}")
    print(f"  Train size   : {X_train_scaled.shape[0]:,} lignes")
    print(f"  Cross-val    : 3-fold stratifié")

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    models_config = {
        "LogisticRegression":   LogisticRegression(max_iter=500, random_state=42, n_jobs=-1),
        "RandomForest":         RandomForestClassifier(n_estimators=50, max_depth=8,
                                                        random_state=42, n_jobs=-1),
        "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=80, max_depth=6,
                                                                 learning_rate=0.1, random_state=42),
        "LinearSVM":            CalibratedClassifierCV(LinearSVC(max_iter=1000, random_state=42), cv=3),
        "XGBoost":              xgb.XGBClassifier(max_depth=5, n_estimators=80, learning_rate=0.1,
                                                    subsample=0.8, colsample_bytree=0.8,
                                                    random_state=42, verbosity=0,
                                                    eval_metric="mlogloss", n_jobs=-1),
    }

    trained_models = {}
    all_results    = {}

    for model_name, model in models_config.items():
        print(f"\n{'─'*60}")
        print(f"  Modèle : {model_name}")
        t0 = time.time()
        cv_res = cross_validate(model, X_train_scaled, y_train, cv=skf,
                                scoring=["accuracy", "precision_weighted", "recall_weighted", "f1_weighted"],
                                n_jobs=-1)
        acc_cv  = cv_res["test_accuracy"].mean()
        std_cv  = cv_res["test_accuracy"].std()
        prec_cv = cv_res["test_precision_weighted"].mean()
        rec_cv  = cv_res["test_recall_weighted"].mean()
        f1_cv   = cv_res["test_f1_weighted"].mean()
        print(f"  CV Accuracy  : {acc_cv*100:.2f}% (±{std_cv*100:.2f}%)  |  CV F1 : {f1_cv*100:.2f}%")

        model.fit(X_train_scaled, y_train)
        y_pred    = model.predict(X_test_scaled)
        acc_test  = accuracy_score(y_test, y_pred)
        prec_test = precision_score(y_test, y_pred, average="weighted", zero_division=0)
        rec_test  = recall_score(y_test,  y_pred, average="weighted", zero_division=0)
        f1_test   = f1_score(y_test,  y_pred, average="weighted", zero_division=0)
        print(f"  Test Accuracy : {acc_test*100:.2f}%  |  Temps : {time.time()-t0:.1f}s")

        trained_models[model_name] = model
        all_results[model_name] = {
            "model": model, "y_pred": y_pred,
            "cv_accuracy": acc_cv, "cv_std": std_cv,
            "cv_precision": prec_cv, "cv_recall": rec_cv, "cv_f1": f1_cv,
            "test_accuracy": acc_test, "test_precision": prec_test,
            "test_recall": rec_test, "test_f1": f1_test,
        }

    # Sauvegarder
    _data_dir  = os.path.dirname(os.path.abspath(data_path))
    models_dir = os.path.normpath(os.path.join(_data_dir, "..", "03_Data_Science", "models"))
    os.makedirs(models_dir, exist_ok=True)

    for name, model in trained_models.items():
        with open(os.path.join(models_dir, f"{name}.pkl"), "wb") as fp:
            pickle.dump(model, fp)
    with open(os.path.join(models_dir, "scaler.pkl"), "wb") as fp:
        pickle.dump(scaler, fp)
    with open(os.path.join(models_dir, "label_encoder.pkl"), "wb") as fp:
        pickle.dump(le, fp)
    with open(os.path.join(models_dir, "feature_cols.pkl"), "wb") as fp:
        pickle.dump(feature_cols, fp)

    print("\n" + "="*80)
    print("RÉCAPITULATIF — TOUS LES MODÈLES (5 CLASSES)")
    print("="*80)
    print(f"  {'Modèle':<25} {'CV Acc':>9}  {'±':>5}  {'Test Acc':>9}  {'CV F1':>8}")
    print(f"  {'─'*65}")
    for name, r in sorted(all_results.items(), key=lambda x: -x[1]["cv_accuracy"]):
        print(f"  {name:<25} {r['cv_accuracy']*100:>8.2f}%  {r['cv_std']*100:>4.2f}%  "
              f"{r['test_accuracy']*100:>8.2f}%  {r['cv_f1']*100:>7.2f}%")

    best_model_name = max(all_results, key=lambda x: all_results[x]["cv_accuracy"])
    best_model      = trained_models[best_model_name]
    best_accuracy   = all_results[best_model_name]["cv_accuracy"]
    accuracy        = all_results[best_model_name]["test_accuracy"]
    precision       = all_results[best_model_name]["test_precision"]
    recall          = all_results[best_model_name]["test_recall"]
    f1              = all_results[best_model_name]["test_f1"]
    y_pred_best     = all_results[best_model_name]["y_pred"]

    print(f"\n  MEILLEUR MODÈLE : {best_model_name}  (CV Accuracy : {best_accuracy*100:.2f}%)")
    print(f"  Classes         : {list(le.classes_)}")
    print(f"  Sauvegardé dans : {os.path.abspath(models_dir)}")


ÉTAPE 3 : ENTRAÎNEMENT (5 CLASSES — OPTIMISÉ POUR LA VITESSE)
  Classes      : ['Boom', 'Crise', 'Croissance', 'Declin', 'Stable']
  Train size   : 32,000 lignes
  Cross-val    : 3-fold stratifié

────────────────────────────────────────────────────────────
  Modèle : LogisticRegression


  CV Accuracy  : 82.57% (±0.10%)  |  CV F1 : 82.52%


  Test Accuracy : 82.10%  |  Temps : 6.3s

────────────────────────────────────────────────────────────
  Modèle : RandomForest


  CV Accuracy  : 80.14% (±0.35%)  |  CV F1 : 79.80%


  Test Accuracy : 79.57%  |  Temps : 5.0s

────────────────────────────────────────────────────────────
  Modèle : HistGradientBoosting


...

  Test Accuracy : 82.11%  |  Temps : 11.1s

────────────────────────────────────────────────────────────
  Modèle : LinearSVM


  CV Accuracy  : 71.88% (±0.17%)  |  CV F1 : 69.99%


  Test Accuracy : 70.70%  |  Temps : 4.3s

────────────────────────────────────────────────────────────
  Modèle : XGBoost


  CV Accuracy  : 82.23% (±0.05%)  |  CV F1 : 82.21%


  Test Accuracy : 82.05%  |  Temps : 4.1s

RÉCAPITULATIF — TOUS LES MODÈLES (5 CLASSES)
  Modèle                       CV Acc      ±   Test Acc     CV F1
  ─────────────────────────────────────────────────────────────────
  LogisticRegression           82.57%  0.10%     82.10%    82.52%
  XGBoost                      82.23%  0.05%     82.05%    82.21%
  HistGradientBoosting         81.84%  0.12%     82.11%    81.83%
  RandomForest                 80.14%  0.35%     79.57%    79.80%
  LinearSVM                    71.88%  0.17%     70.70%    69.99%

  MEILLEUR MODÈLE : LogisticRegression  (CV Accuracy : 82.57%)
  Classes         : ['Boom', 'Crise', 'Croissance', 'Declin', 'Stable']
  Sauvegardé dans : c:\Users\tarek\Downloads\MsprBigData\MSPR_Final\MSPR\03_Data_Science\models


In [4]:
# ============================================================================
# 4. MÉTRIQUES DÉTAILLÉES — TOUS LES MODÈLES
# ============================================================================
print("\n" + "="*80)
print("ÉTAPE 4 : ANALYSE DÉTAILLÉE — TOUS LES MODÈLES")
print("="*80)

for model_name, r in sorted(all_results.items(), key=lambda x: -x[1]["cv_accuracy"]):
    print(f"\n{'─'*60}")
    marker = "  ← MEILLEUR" if model_name == best_model_name else ""
    print(f"  {model_name}{marker}")
    print(f"  Accuracy : {r['test_accuracy']*100:.2f}%  |  Precision : {r['test_precision']*100:.2f}%  "
          f"|  Recall : {r['test_recall']*100:.2f}%  |  F1 : {r['test_f1']*100:.2f}%")
    print(classification_report(y_test, r["y_pred"], target_names=le.classes_))
    cm = confusion_matrix(y_test, r["y_pred"])
    print(f"  Matrice de confusion :\n{cm}")



ÉTAPE 4 : ANALYSE DÉTAILLÉE — TOUS LES MODÈLES

────────────────────────────────────────────────────────────
  LogisticRegression  ← MEILLEUR
  Accuracy : 82.10%  |  Precision : 82.16%  |  Recall : 82.10%  |  F1 : 82.06%
              precision    recall  f1-score   support

        Boom       0.89      0.81      0.85       800
       Crise       0.88      0.79      0.83       800
  Croissance       0.79      0.80      0.79      1600
      Declin       0.77      0.74      0.75      1600
      Stable       0.83      0.88      0.86      3200

    accuracy                           0.82      8000
   macro avg       0.83      0.81      0.82      8000
weighted avg       0.82      0.82      0.82      8000

  Matrice de confusion :
[[ 648    0  152    0    0]
 [   0  634    0  166    0]
 [  83    0 1276    0  241]
 [   0   85    0 1191  324]
 [   0    0  183  198 2819]]

────────────────────────────────────────────────────────────
  XGBoost
  Accuracy : 82.05%  |  Precision : 82.17%  |  Reca

In [5]:
# ============================================================================
# 5. PRÉDICTIONS PAR NIVEAU GÉOGRAPHIQUE — TOUS LES MODÈLES
# ============================================================================
import unicodedata

print("\n" + "="*80)
print("ÉTAPE 5 : PRÉDICTIONS PAR NIVEAU GÉOGRAPHIQUE")
print("="*80)

df_orig    = df.copy()

# Mapping 5 classes → candidat politique
# Boom/Croissance/Stable → zones favorables au centre (Macron)
# Declin/Crise → zones défavorisées (Le Pen)
WINNER_MAP = {
    "Boom":       "MACRON",
    "Croissance": "MACRON",
    "Stable":     "MACRON",
    "Declin":     "LE PEN",
    "Crise":      "LE PEN",
}

def clean_name(name):
    if not isinstance(name, str): return ""
    name = "".join(c for c in unicodedata.normalize("NFD", name) if unicodedata.category(c) != "Mn")
    return name.upper().strip().replace("-", " ")

def predict_all_models(group_X):
    """Applique chaque modèle sur la moyenne des features du groupe."""
    if len(group_X) == 0: return {}
    features        = group_X.mean().values.reshape(1, -1)
    features_scaled = scaler.transform(features)
    results = {}
    for mname, model in trained_models.items():
        pred_idx = model.predict(features_scaled)[0]
        proba    = model.predict_proba(features_scaled)[0]
        pd_dict  = {le.classes_[i]: float(proba[i]) for i in range(len(le.classes_))}
        # Macron = Boom + Croissance + Stable | Le Pen = Declin + Crise
        prob_mac = pd_dict.get("Boom", 0) + pd_dict.get("Croissance", 0) + pd_dict.get("Stable", 0)
        prob_lp  = pd_dict.get("Declin", 0) + pd_dict.get("Crise", 0)
        eco_cls  = le.inverse_transform([pred_idx])[0]
        results[mname] = {
            "eco_class":    eco_cls,
            "candidate":    WINNER_MAP.get(eco_cls, "MACRON"),
            "confidence":   float(np.max(proba)),
            "proba_macron": round(prob_mac * 100, 2),
            "proba_lepen":  round(prob_lp  * 100, 2),
        }
    return results

# ── Détection des colonnes géographiques dans le dataset réel ─────────────────
dept_col = next(
    (c for c in df_orig.columns if "partement" in c.lower() and "libell" in c.lower()),
    next((c for c in df_orig.columns
          if "departement" in c.lower() and "code" not in c.lower()), None)
)
canton_col = next(
    (c for c in df_orig.columns if "canton" in c.lower() and "libell" in c.lower()),
    None
)
# Fallback: utilise le code département si aucune colonne libellé trouvée
if dept_col is None and "code_departement" in df_orig.columns:
    dept_col = "code_departement"

if dept_col:
    df_orig["parsed_departement"] = df_orig[dept_col].apply(clean_name)
    print(f"  Colonne département détectée : '{dept_col}'")
if canton_col:
    df_orig["parsed_canton"] = df_orig[canton_col].apply(clean_name)
    print(f"  Colonne canton détectée     : '{canton_col}'")

# ── NIVEAU RÉGION ─────────────────────────────────────────────────────────────
print("\n RÉGION — NOUVELLE-AQUITAINE")
print("─"*70)
region_preds = predict_all_models(X_clean)
for mname, r in region_preds.items():
    mark = "  ← meilleur" if mname == best_model_name else ""
    print(f"  {mname:<22} → {r['candidate']:<10} [{r['eco_class']:<10}]  Macron:{r['proba_macron']:>5.1f}%  LePen:{r['proba_lepen']:>5.1f}%  conf:{r['confidence']*100:.1f}%{mark}")

# ── NIVEAU DÉPARTEMENT ────────────────────────────────────────────────────────
dept_geo_preds = {}
if "parsed_departement" in df_orig.columns:
    print(f"\n DÉPARTEMENTS — prédiction via {best_model_name} (tous modèles stockés)")
    print("─"*70)
    for dept in sorted([d for d in df_orig["parsed_departement"].unique() if d]):
        mask   = df_orig["parsed_departement"] == dept
        dept_X = X_clean.loc[mask]
        preds  = predict_all_models(dept_X)
        dept_geo_preds[dept] = preds
        bp = preds.get(best_model_name, {})
        print(f"  {dept:<28} [{best_model_name}] → {bp.get('candidate','?'):<8} [{bp.get('eco_class','?'):<10}]  "
              f"Macron:{bp.get('proba_macron',0):>5.1f}%  LePen:{bp.get('proba_lepen',0):>5.1f}%")

# ── NIVEAU CANTON ─────────────────────────────────────────────────────────────
canton_geo_preds = {}
if "parsed_canton" in df_orig.columns:
    top_cantons = [c for c in df_orig["parsed_canton"].value_counts().index if c]
    print(f"\n TOP 10 CANTONS — {best_model_name}")
    print("─"*70)
    for canton in top_cantons:
        mask     = df_orig["parsed_canton"] == canton
        canton_X = X_clean.loc[mask]
        preds    = predict_all_models(canton_X)
        canton_geo_preds[canton] = preds
        bp = preds.get(best_model_name, {})
        print(f"  {str(canton)[:32]:<34} → {bp.get('candidate','?'):<8} [{bp.get('eco_class','?'):<10}]  Macron:{bp.get('proba_macron',0):>5.1f}%")


ÉTAPE 5 : PRÉDICTIONS PAR NIVEAU GÉOGRAPHIQUE


  Colonne département détectée : 'Libellé du département'

  Colonne canton détectée     : 'Libellé du canton'

 RÉGION — NOUVELLE-AQUITAINE
──────────────────────────────────────────────────────────────────────


  LogisticRegression     → MACRON     [Stable    ]  Macron: 98.4%  LePen:  1.6%  conf:97.1%  ← meilleur
  RandomForest           → MACRON     [Stable    ]  Macron: 80.2%  LePen: 19.8%  conf:56.1%
  HistGradientBoosting   → MACRON     [Stable    ]  Macron: 82.8%  LePen: 17.2%  conf:68.7%
  LinearSVM              → MACRON     [Stable    ]  Macron: 80.3%  LePen: 19.7%  conf:61.3%
  XGBoost                → MACRON     [Stable    ]  Macron: 85.3%  LePen: 14.7%  conf:59.3%

 DÉPARTEMENTS — prédiction via LogisticRegression (tous modèles stockés)
──────────────────────────────────────────────────────────────────────
  CHARENTE                     [LogisticRegression] → MACRON   [Stable    ]  Macron: 99.1%  LePen:  0.9%


  CHARENTE MARITIME            [LogisticRegression] → MACRON   [Stable    ]  Macron: 99.8%  LePen:  0.2%
  CORREZE                      [LogisticRegression] → MACRON   [Stable    ]  Macron: 99.6%  LePen:  0.4%


...

  CARLUX                             → LE PEN   [Declin    ]  Macron:  3.5%
  BRIOUX SUR BOUTONNE                → LE PEN   [Declin    ]  Macron:  1.7%


  MONEIN                             → LE PEN   [Crise     ]  Macron:  0.0%
  SUD BERGERACOIS                    → MACRON   [Stable    ]  Macron: 99.8%


  LIMOGES 9                          → MACRON   [Boom      ]  Macron:100.0%
  ARS EN RE                          → MACRON   [Stable    ]  Macron: 99.9%


  LA TESTE                           → MACRON   [Stable    ]  Macron: 96.1%
  JAUNAY CLAN                        → MACRON   [Stable    ]  Macron: 79.1%


  GUERET SUD EST                     → MACRON   [Croissance]  Macron:100.0%


In [6]:
# ============================================================================
# 6. RAPPORT FINAL — TOUS LES MODÈLES
# ============================================================================
print("\n" + "="*80)
print("RAPPORT FINAL — NOUVELLE-AQUITAINE")
print("="*80)

print(f"\n  Enregistrements : {len(df):,}")
print(f"  Features        : {len(feature_cols)}")
print(f"  Classes         : {list(le.classes_)}")

print(f"\n  TOUS LES MODÈLES :")
print(f"  {'Modèle':<22} {'CV Acc':>9}  {'±':>5}  {'Test Acc':>9}  {'F1':>8}")
print(f"  {'─'*62}")
for name, r in sorted(all_results.items(), key=lambda x: -x[1]["cv_accuracy"]):
    marker = "  ← MEILLEUR" if name == best_model_name else ""
    print(f"  {name:<22} {r['cv_accuracy']*100:>8.2f}%  {r['cv_std']*100:>4.2f}%  "
          f"{r['test_accuracy']*100:>8.2f}%  {r['cv_f1']*100:>7.2f}%{marker}")

best_r = region_preds.get(best_model_name, {})
print(f"\n  PRÉDICTION RÉGIONALE (meilleur modèle : {best_model_name}) :")
print(f"  Candidat : {best_r.get('candidate', '?')}  |  "
      f"Macron : {best_r.get('proba_macron', 0):.2f}%  |  Le Pen : {best_r.get('proba_lepen', 0):.2f}%")

print("\n  COMPARAISON TOUS MODÈLES — RÉGION :")
for name, r in region_preds.items():
    mark = "  ← meilleur" if name == best_model_name else ""
    print(f"    {name:<22} → {r['candidate']:<10} (Macron:{r['proba_macron']:.1f}%  LePen:{r['proba_lepen']:.1f}%){mark}")



RAPPORT FINAL — NOUVELLE-AQUITAINE

  Enregistrements : 40,000
  Features        : 27
  Classes         : ['Boom', 'Crise', 'Croissance', 'Declin', 'Stable']

  TOUS LES MODÈLES :
  Modèle                    CV Acc      ±   Test Acc        F1
  ──────────────────────────────────────────────────────────────
  LogisticRegression        82.57%  0.10%     82.10%    82.52%  ← MEILLEUR
  XGBoost                   82.23%  0.05%     82.05%    82.21%
  HistGradientBoosting      81.84%  0.12%     82.11%    81.83%
  RandomForest              80.14%  0.35%     79.57%    79.80%
  LinearSVM                 71.88%  0.17%     70.70%    69.99%

  PRÉDICTION RÉGIONALE (meilleur modèle : LogisticRegression) :
  Candidat : MACRON  |  Macron : 98.44%  |  Le Pen : 1.56%

  COMPARAISON TOUS MODÈLES — RÉGION :
    LogisticRegression     → MACRON     (Macron:98.4%  LePen:1.6%)  ← meilleur
    RandomForest           → MACRON     (Macron:80.2%  LePen:19.8%)
    HistGradientBoosting   → MACRON     (Macron:82.8% 

In [7]:
# ============================================================================
# 7. ANALYSE DES FEATURE IMPORTANCES
# ============================================================================
print("\n" + "="*80)
print("ÉTAPE 6 : IMPORTANCE DES FEATURES")
print("="*80)

# Récupérer les importances du meilleur modèle
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print(f"\n🔝 TOP 15 FEATURES LES PLUS IMPORTANTES :")
    print("-"*80)
    for idx, row in feature_importance_df.head(15).iterrows():
        bar = "█" * int(row['Importance'] * 100)
        print(f"   {row['Feature']:30s} : {bar} {row['Importance']*100:5.2f}%")
        
    print(f"\n📊 Statistiques d'importance :")
    print(f"   - Somme des importances : {importances.sum():.4f}")
    print(f"   - Max importance : {importances.max():.4f}")
    print(f"   - Min importance : {importances.min():.4f}")
    print(f"   - Moyenne : {importances.mean():.4f}")
else:
    print("\n   ⚠️  Ce modèle n'expporte pas les importances")


ÉTAPE 6 : IMPORTANCE DES FEATURES

   ⚠️  Ce modèle n'expporte pas les importances


In [8]:
# ============================================================================
# 8. EXPORT JSON — PRÉDICTIONS DE TOUS LES MODÈLES
# ============================================================================
import json, os

print("\n" + "="*80)
print("ÉTAPE 8 : EXPORT JSON — TOUS LES MODÈLES")
print("="*80)

REAL_WINNERS_2022 = {
    "CHARENTE": "MACRON", "CHARENTE MARITIME": "MACRON", "CORREZE": "MACRON",
    "CREUSE": "MACRON",   "DORDOGNE": "MACRON",          "GIRONDE": "MACRON",
    "LANDES": "MACRON",   "LOT ET GARONNE": "LE PEN",    "PYRENEES ATLANTIQUES": "MACRON",
    "DEUX SEVRES": "MACRON", "VIENNE": "MACRON",         "HAUTE VIENNE": "MACRON",
}

def build_entry(entity, geo_preds, real_winner):
    best_pred = geo_preds.get(best_model_name, {})
    return {
        "entity":       entity,
        "real":         real_winner,
        "best_model":   best_model_name,
        "predicted":    best_pred.get("candidate", "MACRON"),
        "is_correct":   best_pred.get("candidate", "MACRON") == real_winner,
        "proba_macron": best_pred.get("proba_macron", 0),
        "proba_lepen":  best_pred.get("proba_lepen",  0),
        "conf":         f"{best_pred.get('confidence', 0)*100:.0f}%",
        "predictions_by_model": {
            mname: {
                "predicted":    r.get("candidate", "MACRON"),
                "eco_class":    r.get("eco_class",  "?"),
                "proba_macron": r.get("proba_macron", 0),
                "proba_lepen":  r.get("proba_lepen",  0),
                "confidence":   f"{r.get('confidence', 0)*100:.0f}%",
            }
            for mname, r in geo_preds.items()
        },
    }

region_entry   = build_entry("NOUVELLE AQUITAINE", region_preds, "MACRON")
dept_entries   = [build_entry(dept, preds, REAL_WINNERS_2022.get(dept, "MACRON"))
                  for dept, preds in dept_geo_preds.items()]
canton_entries = [build_entry(str(c), preds, "MACRON") for c, preds in canton_geo_preds.items()]

models_metrics = {
    name: {
        "cv_accuracy":    round(r["cv_accuracy"]   * 100, 2),
        "cv_std":         round(r["cv_std"]         * 100, 2),
        "test_accuracy":  round(r["test_accuracy"]  * 100, 2),
        "test_precision": round(r["test_precision"] * 100, 2),
        "test_recall":    round(r["test_recall"]    * 100, 2),
        "test_f1":        round(r["test_f1"]        * 100, 2),
    }
    for name, r in all_results.items()
}

nb_macron_pred = sum(1 for d in dept_entries if d["predicted"] == "MACRON")
nb_lepen_pred  = sum(1 for d in dept_entries if d["predicted"] == "LE PEN")
dept_acc       = sum(1 for d in dept_entries if d["is_correct"]) / len(dept_entries) * 100 if dept_entries else 0

export_data = {
    "summary": {
        "region_name":         "Nouvelle-Aquitaine",
        "best_model":          best_model_name,
        "best_model_accuracy": round(accuracy * 100, 2),
        "dept_accuracy":       round(dept_acc, 1),
        "models_list":         list(trained_models.keys()),
        "total_records":       len(df_orig),
    },
    "models_metrics": models_metrics,
    "political_predicted": [
        {"party": "MACRON",  "count": nb_macron_pred, "color": "#0055A4"},
        {"party": "LE PEN",  "count": nb_lepen_pred,  "color": "#8B0000"},
    ],
    "political_real": [
        {"party": "MACRON", "count": sum(1 for v in REAL_WINNERS_2022.values() if v == "MACRON"), "color": "#0055A4"},
        {"party": "LE PEN", "count": sum(1 for v in REAL_WINNERS_2022.values() if v == "LE PEN"),  "color": "#8B0000"},
    ],
    "levels": {
        "region":      [region_entry],
        "departement": dept_entries,
        "canton":      canton_entries,
        "commune":     [],
    },
}

_data_dir   = os.path.dirname(os.path.abspath(data_path))
export_dir  = os.path.normpath(os.path.join(_data_dir, "..", "03_Data_Science", "Visualisation", "data"))
export_path = os.path.join(export_dir, "predictions.json")
os.makedirs(os.path.dirname(export_path), exist_ok=True)

with open(export_path, "w", encoding="utf-8") as fp:
    json.dump(export_data, fp, ensure_ascii=False, indent=2)

print(f"  Fichier exporté : {os.path.abspath(export_path)}")
print(f"  Modèles inclus  : {list(trained_models.keys())}")
print(f"  Départements    : {len(dept_entries)}")
print(f"  Cantons         : {len(canton_entries)}")
print(f"  Dept accuracy   : {dept_acc:.1f}%")
for name, m in models_metrics.items():
    print(f"    {name:<22}  CV:{m['cv_accuracy']:.2f}%  Test:{m['test_accuracy']:.2f}%  F1:{m['test_f1']:.2f}%")



ÉTAPE 8 : EXPORT JSON — TOUS LES MODÈLES


  Fichier exporté : c:\Users\tarek\Downloads\MsprBigData\MSPR_Final\MSPR\03_Data_Science\Visualisation\data\predictions.json
  Modèles inclus  : ['LogisticRegression', 'RandomForest', 'HistGradientBoosting', 'LinearSVM', 'XGBoost']
  Départements    : 12
  Cantons         : 627
  Dept accuracy   : 91.7%
    LogisticRegression      CV:82.57%  Test:82.10%  F1:82.06%
    RandomForest            CV:80.14%  Test:79.57%  F1:79.22%
    HistGradientBoosting    CV:81.84%  Test:82.11%  F1:82.10%
    LinearSVM               CV:71.88%  Test:70.70%  F1:68.67%
    XGBoost                 CV:82.23%  Test:82.05%  F1:82.03%
